# ETH 1-Year DCA Backtest (Hourly Check)

策略规则：
- 使用数据：`/Users/dennis/projects/TradingTutorial/data/eth_usdt_1m_1y.csv`
- 每小时检查一次市场价（每小时最后一个 1m close）
- 若当前小时价格 <= 前一天收盘价 * 0.99，则买入 `100 USDT` 的 ETH
- 回测统计：累计投入、最终市值、PnL、收益率、夏普率、最大回撤


In [ ]:
import csv
from datetime import datetime, timedelta
from math import sqrt

DATA_PATH = '/Users/dennis/projects/TradingTutorial/data/eth_usdt_1m_1y.csv'
BUY_USDT = 100.0
DROP_THRESHOLD = 0.01  # 1%
SHARPE_ANNUAL_FACTOR = 365


In [ ]:
daily_close = {}
hourly_last = {}
first_ts = None
last_ts = None
last_close = None

with open(DATA_PATH, 'r', newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        ts = datetime.fromisoformat(row['datetime'])
        px = float(row['close'])

        if first_ts is None:
            first_ts = ts
        last_ts = ts
        last_close = px

        day = ts.date()
        hour = ts.replace(minute=0, second=0, microsecond=0)

        # overwrite so we keep last price in the bucket
        daily_close[day] = px
        hourly_last[hour] = px

print('Data range:', first_ts, '->', last_ts)
print('Days:', len(daily_close), 'Hours:', len(hourly_last))

In [ ]:
eth_qty = 0.0
invested = 0.0
buy_count = 0

trade_log = []
hour_keys = sorted(hourly_last.keys())

for h in hour_keys:
    day = h.date()
    prev_day = day - timedelta(days=1)
    if prev_day not in daily_close:
        continue

    prev_close = daily_close[prev_day]
    px = hourly_last[h]

    if px <= prev_close * (1 - DROP_THRESHOLD):
        qty = BUY_USDT / px
        eth_qty += qty
        invested += BUY_USDT
        buy_count += 1
        trade_log.append((h, px, prev_close, qty, eth_qty, invested))

final_value = eth_qty * last_close if last_close is not None else 0.0
final_pnl = final_value - invested
total_return = (final_value / invested - 1.0) if invested > 0 else float('nan')

print(f'Buy Count:      {buy_count}')
print(f'Total Invested: {invested:,.2f} USDT')
print(f'Final Value:    {final_value:,.2f} USDT')
print(f'Final PnL:      {final_pnl:,.2f} USDT')
print(f'Total Return:   {total_return * 100:.2f}%')

In [ ]:
# Build daily NAV (flow-adjusted proxy): NAV = portfolio_value / cumulative_invested
daily_nav = []
daily_dates = []

eth = 0.0
inv = 0.0
hour_ptr = 0

sorted_days = sorted(daily_close.keys())
for day in sorted_days:
    prev_day = day - timedelta(days=1)
    prev_close = daily_close.get(prev_day)

    # process hourly checks for this day
    while hour_ptr < len(hour_keys) and hour_keys[hour_ptr].date() == day:
        h = hour_keys[hour_ptr]
        px = hourly_last[h]
        if prev_close is not None and px <= prev_close * (1 - DROP_THRESHOLD):
            eth += BUY_USDT / px
            inv += BUY_USDT
        hour_ptr += 1

    if inv > 0:
        nav = (eth * daily_close[day]) / inv
        daily_nav.append(nav)
        daily_dates.append(day)

daily_rets = []
for i in range(1, len(daily_nav)):
    prev = daily_nav[i - 1]
    cur = daily_nav[i]
    if prev != 0:
        daily_rets.append(cur / prev - 1.0)

if len(daily_rets) > 1:
    mean_r = sum(daily_rets) / len(daily_rets)
    var_r = sum((r - mean_r) ** 2 for r in daily_rets) / (len(daily_rets) - 1)
    std_r = var_r ** 0.5
    sharpe = (mean_r / std_r) * sqrt(SHARPE_ANNUAL_FACTOR) if std_r != 0 else float('nan')
else:
    sharpe = float('nan')

# max drawdown on NAV
max_dd = 0.0
if daily_nav:
    peak = daily_nav[0]
    for v in daily_nav:
        if v > peak:
            peak = v
        dd = v / peak - 1.0
        if dd < max_dd:
            max_dd = dd

print(f'Sharpe Ratio (annualized): {sharpe:.4f}')
print(f'Max Drawdown:              {max_dd * 100:.2f}%')
print(f'Daily NAV points:          {len(daily_nav)}')

In [ ]:
# Optional plotting (if matplotlib is installed)
try:
    import matplotlib.pyplot as plt

    if trade_log:
        trade_times = [x[0] for x in trade_log]
        trade_prices = [x[1] for x in trade_log]
    else:
        trade_times, trade_prices = [], []

    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

    axes[0].plot(sorted(hourly_last.keys()), [hourly_last[k] for k in sorted(hourly_last.keys())], linewidth=0.8, label='Hourly Close')
    if trade_times:
        axes[0].scatter(trade_times, trade_prices, s=8, alpha=0.7, label='DCA Buys')
    axes[0].set_title('ETH Hourly Price & DCA Buy Points')
    axes[0].set_ylabel('Price')
    axes[0].legend()

    axes[1].plot(daily_dates, daily_nav, linewidth=1.0, color='tab:green')
    axes[1].set_title('Daily NAV (Portfolio / Invested)')
    axes[1].set_ylabel('NAV')
    axes[1].set_xlabel('Date')

    plt.tight_layout()
    plt.show()
except Exception as e:
    print('Plot skipped:', e)